# PyTorch ATen Hardware Support Matrix Exporter

This notebook is designed to run in **Google Colab**. It automates installing multiple versions of PyTorch (2.5.0, 2.4.0, 2.3.0), extracting the ATen hardware support matrix, and saving the outputs directly to your Google Drive.

### Why do we need to restart the runtime?
When you install a different version of PyTorch via `pip`, the running Python process in Google Colab still keeps the old PyTorch version loaded in memory. To load the newly installed version, we must restart the Python runtime. Calling `os.kill(os.getpid(), 9)` terminates the current process, forcing Google Colab to automatically restart the runtime shell without clearing the disk (keeping the installed package and mounted Google Drive intact).

### Workaround for State Loss
Since restarting the runtime clears Python's memory (imported modules, functions, and variables are wiped), we cannot define the exporter function inside the notebook memory. Instead, we write the exporter logic to a standalone file `/content/exporter.py` on the local disk. This file persists across runtime restarts, allowing us to run it as a shell command `!python /content/exporter.py` for each PyTorch version.

### Workflow Steps:
1. **Mount Google Drive** to store the exported JSON matrices.
2. **Write the core exporter script** to `/content/exporter.py`.
3. **Run version-specific workflows** sequentially:
   - Install PyTorch `2.5.0` -> Restart Runtime -> Run Exporter.
   - Install PyTorch `2.4.0` -> Restart Runtime -> Run Exporter.
   - Install PyTorch `2.3.0` -> Restart Runtime -> Run Exporter.

## Section 1: Mount Google Drive

Mount your Google Drive to save the exported files securely. We will create a directory named `pytorch_aten_exports` in your Google Drive root.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive to /content/drive
drive.mount('/content/drive')

# Setup output directory
drive_out_dir = '/content/drive/MyDrive/pytorch_aten_exports'
os.makedirs(drive_out_dir, exist_ok=True)
print(f"Target directory initialized at: {drive_out_dir}")

## Section 2: Write Core Exporter Script

We write the exporter to a file `/content/exporter.py`. This script uses PyTorch internal dispatch APIs `torch._C._dispatch_get_all_op_names()` and `torch._C._dispatch_has_kernel_for_dispatch_key()` to construct the operator support matrix.

In [ ]:
%%writefile /content/exporter.py
import torch
import json
import os
import argparse

def main():
    parser = argparse.ArgumentParser(description="Export PyTorch ATen operator dispatch table")
    parser.add_argument("--output", type=str, required=True, help="Path to save the JSON output")
    args = parser.parse_args()
    
    torch_version = torch.__version__
    print(f"Starting export for PyTorch {torch_version}...")
    
    # List of primary dispatch keys of interest
    dispatch_keys = [
        "CPU",
        "CUDA",
        "MPS",
        "Meta",
        "CompositeImplicitAutograd",
        "CompositeExplicitAutograd",
        "CompositeExplicitAutogradNonFunctional",
        "AutogradCPU",
        "AutogradCUDA",
        "XLA",
        "IPU",
        "HPU"
    ]
    
    # Get all op names via PyTorch internal dispatch registry
    try:
        op_names = torch._C._dispatch_get_all_op_names()
    except AttributeError:
        # Fallback to scanning torch.ops namespaces if private API changes
        op_names = []
        for ns_name in dir(torch.ops):
            ns = getattr(torch.ops, ns_name)
            for op_name in dir(ns):
                op_names.append(f"{ns_name}::{op_name}")
                
    results = []
    has_kernel_fn = getattr(torch._C, "_dispatch_has_kernel_for_dispatch_key", None)
    
    for op_name in sorted(list(set(op_names))):
        op_info = {
            "name": op_name,
            "kernels": {}
        }
        for key in dispatch_keys:
            if has_kernel_fn:
                try:
                    op_info["kernels"][key] = has_kernel_fn(op_name, key)
                except Exception:
                    op_info["kernels"][key] = None
            else:
                op_info["kernels"][key] = None
        results.append(op_info)
        
    out_data = {
        "pytorch_version": torch_version,
        "operators": results
    }
    
    os.makedirs(os.path.dirname(args.output), exist_ok=True)
    with open(args.output, "w", encoding="utf-8") as f:
        json.dump(out_data, f, indent=2)
        
    print(f"Successfully exported {len(results)} operators to {args.output}")

if __name__ == "__main__":
    main()

## Section 3: PyTorch Version-specific Workflows

Please execute the cells below version by version. 
**Crucial**: After running the **Restart Runtime** cell for any version, wait for Google Colab to reconnect before triggering the corresponding **Run Exporter** cell.

---
### Workflow for PyTorch 2.5.0

**Step 1: Install PyTorch 2.5.0**

In [ ]:
!pip install -q torch==2.5.0

**Step 2: Restart Runtime**

Run the cell below to restart the runtime. Wait until Colab reconnects.

In [ ]:
import os
os.kill(os.getpid(), 9)

**Step 3: Run Exporter for 2.5.0**

In [ ]:
!python /content/exporter.py --output /content/drive/MyDrive/pytorch_aten_exports/aten_v2.5.0.json

---
### Workflow for PyTorch 2.4.0

**Step 1: Install PyTorch 2.4.0**

In [ ]:
!pip install -q torch==2.4.0

**Step 2: Restart Runtime**

Run the cell below to restart the runtime. Wait until Colab reconnects.

In [ ]:
import os
os.kill(os.getpid(), 9)

**Step 3: Run Exporter for 2.4.0**

In [ ]:
!python /content/exporter.py --output /content/drive/MyDrive/pytorch_aten_exports/aten_v2.4.0.json

---
### Workflow for PyTorch 2.3.0

**Step 1: Install PyTorch 2.3.0**

In [ ]:
!pip install -q torch==2.3.0

**Step 2: Restart Runtime**

Run the cell below to restart the runtime. Wait until Colab reconnects.

In [ ]:
import os
os.kill(os.getpid(), 9)

**Step 3: Run Exporter for 2.3.0**

In [ ]:
!python /content/exporter.py --output /content/drive/MyDrive/pytorch_aten_exports/aten_v2.3.0.json